In [14]:
import pandas as pd
from sqlalchemy import create_engine, text, inspect

engine = create_engine('sqlite:///formula1.sqlite')

In [15]:
# SELECT, FROM, LIMIT
#forma 1: directo con pandas 
query = text("""
SELECT * FROM drivers LIMIT 5;
""")
# Esta consulta me permite saber las columnas cuidando no sobrecargar  
df = pd.read_sql_query(query, engine)

query2 = text("""SELECT COUNT(1) FROM drivers;""")
# Esta consulta me permite saber el total de filas
df_count = pd.read_sql_query(query2, engine)

'''
#forma 2: usando engine y connection con with

with engine.connect() as connection:
    result = connection.execute(query)
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
df.columns
'''

'\n#forma 2: usando engine y connection con with\n\nwith engine.connect() as connection:\n    result = connection.execute(query)\n    df = pd.DataFrame(result.fetchall(), columns=result.keys())\ndf.columns\n'

In [16]:
df.columns

Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
       'nationality', 'url'],
      dtype='object')

In [17]:
query3 = text("""SELECT driverId, forename, surname FROM drivers LIMIT 5;""")
df_driver = pd.read_sql_query(query3, engine)

In [18]:
df

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,2,heidfeld,\N,HEI,Nick,Heidfeld,1977-05-10,German,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,http://en.wikipedia.org/wiki/Nico_Rosberg
3,4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,http://en.wikipedia.org/wiki/Fernando_Alonso
4,5,kovalainen,\N,KOV,Heikki,Kovalainen,1981-10-19,Finnish,http://en.wikipedia.org/wiki/Heikki_Kovalainen


In [19]:
df_driver

,driverId,forename,surname
0,1,Lewis,Hamilton
1,2,Nick,Heidfeld
2,3,Nico,Rosberg
3,4,Fernando,Alonso
4,5,Heikki,Kovalainen


In [20]:
# Para generar un entregable guardamos el DataFrame en un archivo CSV 
df_driver.to_csv('drivers1.csv', index=False, sep=';')
# Siempre especificar el separador para evitar problemas con los datos 

In [21]:
# SELECT, FROM, WHERE
# pregunta: ¿Cuantos pilotos son británicos?
query4 = text("""
SELECT driverId, forename, surname FROM drivers WHERE nationality = 'British';
""")
df_driver_british = pd.read_sql_query(query4, engine)

In [22]:
# Otra forma.. 
query5 = text("""SELECT COUNT(1) FROM drivers WHERE nationality = 'British';
""")
df_driver_british_count = pd.read_sql_query(query5, engine)

In [23]:
df_driver_british_count

,COUNT(1)
0,166


In [24]:
# Otra forma  
# ¿Quienes son los pilotos australianos?
query = text("""SELECT * FROM drivers""")
df_all_drivers = pd.read_sql_query(query, engine)

df_all_drivers[df_all_drivers.nationality == 'Australian'][['forename', 'surname']]

,forename,surname
16,Mark,Webber
100,David,Brabham
149,Gary,Brabham
177,Alan,Jones
255,Larry,Perkins
265,Brian,McGuire
266,Vern,Schuppan
285,Warwick,Brown
313,Tim,Schenken
337,David,Walker


In [25]:
# SELECT, FROM, LIMIT
def select_from_limit():
    with engine.connect() as conn:
        result = conn.execute(text("SELECT * FROM drivers LIMIT 5"))
        for row in result:
            print(row)
# SELECT, FROM, WHERE
def select_from_where():
    with engine.connect() as conn:
        result = conn.execute(text("SELECT * FROM drivers WHERE nationality = 'British'"))
        for row in result:
            print(row)  


In [26]:
df_all_drivers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 861 entries, 0 to 860
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   driverId     861 non-null    int64 
 1   driverRef    861 non-null    object
 2   number       861 non-null    object
 3   code         861 non-null    object
 4   forename     861 non-null    object
 5   surname      861 non-null    object
 6   dob          861 non-null    object
 7   nationality  861 non-null    object
 8   url          861 non-null    object
dtypes: int64(1), object(8)
memory usage: 60.7+ KB


In [27]:
# WHERE CON AND, OR, NOT, IN
# ORDER BY

#¿Cuantos corredores son britanicos o franceses?
query = text('''SELECT forename, surname, nationality, dob
                FROM drivers
                --WHERE nationality IN ('British', 'French')
                --WHERE (nationality = 'British' OR nationality = 'French') AND YEAR(dob)<1999
                WHERE (nationality = 'British' OR nationality = 'French') --AND LEFT(dob,4)=1999
               ORDER BY surname
             ''')

df2 = pd.read_sql_query(query, engine)

In [28]:
df2

,forename,surname,nationality,dob
0,George,Abecassis,British,1913-03-21
1,Kenny,Acheson,British,1957-11-27
2,Jack,Aitken,British,1995-09-23
3,Jean,Alesi,French,1964-06-11
4,Philippe,Alliot,French,1954-07-27
...,...,...,...,...
234,Roger,Williamson,British,1948-02-02
235,Justin,Wilson,British,1978-07-31
236,Vic,Wilson,British,1931-04-14
237,Paul,di Resta,British,1986-04-16


In [29]:
import pandas as pd
from sqlalchemy import create_engine, text
# Crear una conexión a la base de datos
engine1 = create_engine('sqlite:///formula1.sqlite')
engine2 = create_engine('sqlite:///f1_database.sqlite')
# Me puedo conectar a varias bases de datos al mismo tiempo 


In [30]:
# Cuales son los pilotos británicos o alemanes nacidos después del 1 de enero de 1985 
# y que su apellido no comience con 'L'
# Clausula LIKE traduce como contiene, NOT LIKE traduce como no contiene
# ASC en ORDER BY significa ascendente, DESC significa descendente
query = text("""
SELECT driverRef, forename, surname, nationality, dob
FROM drivers
WHERE (nationality = 'British' OR nationality = 'German')
  AND dob > '1985-01-01'
  AND forename NOT LIKE 'L%'
ORDER BY dob ASC
LIMIT 20;
""")
df_filtered = pd.read_sql_query(query, engine1) # Le especifico en cual base de datos quiero hacer la consulta 


In [31]:
df_filtered

,driverRef,forename,surname,nationality,dob
0,rosberg,Nico,Rosberg,German,1985-06-27
1,resta,Paul,di Resta,British,1986-04-16
2,vettel,Sebastian,Vettel,German,1987-07-03
3,hulkenberg,Nico,Hülkenberg,German,1987-08-19
4,jolyon_palmer,Jolyon,Palmer,British,1991-01-20
5,chilton,Max,Chilton,British,1991-04-21
6,stevens,Will,Stevens,British,1991-06-28
7,wehrlein,Pascal,Wehrlein,German,1994-10-18
8,aitken,Jack,Aitken,British,1995-09-23
9,russell,George,Russell,British,1998-02-15


In [32]:
# FUNCIONES DE AGREGACION: COUNT, SUM, AVG, MIN, MAX
# ¿Cuantos pilotos hay por nacionalidad? Dame el top 10
query = text("""
SELECT nationality, COUNT(*) as num_drivers
FROM drivers
GROUP BY nationality
ORDER BY num_drivers DESC
LIMIT 10;
""")
df_nationality_counts = pd.read_sql_query(query, engine1)


In [33]:
df_nationality_counts

,nationality,num_drivers
0,British,166
1,American,158
2,Italian,99
3,French,73
4,German,50
5,Brazilian,32
6,Argentine,24
7,Swiss,23
8,South African,23
9,Belgian,23


In [34]:
#¿Cuantos pilotos hay por nacionalidad y por año de nacimiento?
query = text("""
SELECT nationality, strftime('%Y', dob) as birth_year, COUNT(*) as num_drivers
FROM drivers
GROUP BY nationality, birth_year
ORDER BY num_drivers DESC
LIMIT 10;
""")
df_nationality_year_counts = pd.read_sql_query(query, engine1)
    

In [35]:
df_nationality_year_counts

,nationality,birth_year,num_drivers
0,American,1926,13
1,American,1918,8
2,American,1920,8
3,American,1927,8
4,American,1919,7
5,American,1925,7
6,American,1928,7
7,British,1931,7
8,American,1913,6
9,American,1921,6


In [36]:
"""
cdc Change Data Capture
Kafka  
Dama DMbok
Gobierno de Datos Data Governance
Open metadata
""" 


'\ncdc Change Data Capture\nKafka  \nDama DMbok\nGobierno de Datos Data Governance\nOpen metadata\n'

In [37]:
"""
Ejercicio 1: Conteo Total de Carreras
¿Cuántas carreras en total están registradas en la base de datos?
Ejercicio 2: Conteo de Constructores por Nacionalidad
¿Cuántos constructores hay por cada nacionalidad?
Muestra la nacionalidad y el número de constructores, ordenado de la que tiene más a la que tiene menos.
Ejercicio 3: Promedio de Puntos por Estatus de Resultado
¿Cuál es el promedio de puntos que se han otorgado por cada tipo de status (ej. "Finished", "Retired", "Disqualified")?
Muestra el nombre del estatus y su promedio de puntos, ordenado por el promedio de puntos de forma descendente. (Necesitarás un INNER JOIN con la tabla status para el nombre del estatus, puedes adelantarte si te sientes cómodo, o simplemente mostrar statusId por ahora).
Ejercicio 4: Circuitos por País con Múltiples Ubicaciones
¿Cuáles son los países que tienen más de un circuito registrado?
Muestra el país y el número de circuitos, ordenado de forma descendente por el número de circuitos.
Ejercicio 5: Año con Más Carreras
¿Cuál es el año en el que se celebraron la mayor cantidad de carreras?
Muestra el año y el número de carreras para ese año.
Ejercicio 6: Pilotos con más de 50 Carreras
¿Cuáles son los pilotos (por su driverId) que han participado en más de 50 carreras?
Muestra el driverId y el total de carreras en las que participaron, ordenado por el número de carreras de forma descendente. (Puedes contar las entradas en la tabla results por driverId).
Ejercicio 7: Constructor con la Suma Más Alta de Puntos (Resultados)
¿Cuál es el constructorId que ha acumulado la mayor cantidad de puntos en todos los results (no constructor_results)?
Muestra el constructorId y la suma total de puntos.
Ejercicio 8: Meses con Mayor Actividad de Carreras
¿En qué meses se celebran la mayor cantidad de carreras? (Puedes extraer el mes de la columna date de la tabla races. En SQLite, STRFTIME('%m', date) te dará el mes como texto).
Muestra el número del mes y el conteo de carreras, ordenado de forma descendente por el conteo.
Ejercicio 9: Nacionalidades con un Promedio de Edad Superior a 30 Años
¿Cuáles son las nacionalidades de pilotos donde el promedio de edad es superior a 30 años? (Para la edad, puedes calcularla como (STRFTIME('%Y', 'now') - STRFTIME('%Y', dob)), pero es una aproximación. Enfócate en el promedio de la columna dob si es numérico o puedes usar una fecha de referencia como '1990-01-01' para filtrar pilotos).
Muestra la nacionalidad y el conteo de pilotos de esa nacionalidad.
Ordena los resultados por el conteo de pilotos de forma descendente.
Ejercicio 10: Conteo de Vueltas por Carrera
¿Cuántas vueltas diferentes se registraron en cada carrera (por raceId)? (Usa la tabla lap_times).
Muestra el raceId y el conteo de vueltas para cada una.
Muestra solo las primeras 10 carreras con la mayor cantidad de vueltas registradas.
"""

'\nEjercicio 1: Conteo Total de Carreras\n¿Cuántas carreras en total están registradas en la base de datos?\nEjercicio 2: Conteo de Constructores por Nacionalidad\n¿Cuántos constructores hay por cada nacionalidad?\nMuestra la nacionalidad y el número de constructores, ordenado de la que tiene más a la que tiene menos.\nEjercicio 3: Promedio de Puntos por Estatus de Resultado\n¿Cuál es el promedio de puntos que se han otorgado por cada tipo de status (ej. "Finished", "Retired", "Disqualified")?\nMuestra el nombre del estatus y su promedio de puntos, ordenado por el promedio de puntos de forma descendente. (Necesitarás un INNER JOIN con la tabla status para el nombre del estatus, puedes adelantarte si te sientes cómodo, o simplemente mostrar statusId por ahora).\nEjercicio 4: Circuitos por País con Múltiples Ubicaciones\n¿Cuáles son los países que tienen más de un circuito registrado?\nMuestra el país y el número de circuitos, ordenado de forma descendente por el número de circuitos.\nEj

In [42]:
#'''Ejercicio 1: Conteo Total de Carreras
#¿Cuántas carreras en total están registradas en la base de datos?'''

query =  (""" 
            SELECT COUNT(*)
            FROM races""")

df_races = pd.read_sql_query(query, engine1)

In [43]:
df_races

,COUNT(*)
0,1125


In [ ]:
# ejercicio 2¿Cuántos constructores hay por cada nacionalidad?

query = ("""
         SELECT nationality, COUNT(*) AS total_constructores
FROM constructors
GROUP BY nationality
ORDER BY total_constructores DESC;

         """)

df_constructors = pd.read_sql_query(query, engine1)

In [58]:
df_constructors

,nationality,total_constructores
0,British,86
1,American,39
2,Italian,30
3,French,13
4,German,10
5,Swiss,5
6,Japanese,5
7,South African,3
8,Dutch,3
9,Russian,2


In [ ]:
#ejercicio 3¿Cuál es el promedio de puntos que se han otorgado por cada tipo de status (ej. "Finished", "Retired", "Disqualified")? no se sacar el promedio



In [ ]:
# ejercicio 4 ¿Cuáles son los países que tienen más de un circuito registrado? hice la mitad  y tuve que pedir ayuda a chat gpt para terminarlo
query = ("""
         SELECT country, COUNT(*) AS total_circuitos
FROM circuits
GROUP BY country
HAVING COUNT(*) > 1
ORDER BY total_circuitos DESC """)


country_circuit = pd.read_sql_query(query, engine1)

In [61]:
country_circuit

,country,total_circuitos
0,USA,11
1,France,7
2,Spain,6
3,UK,4
4,Portugal,4
5,Italy,4
6,Japan,3
7,Germany,3
8,Canada,3
9,Belgium,3


In [ ]:
# ejercicio 5 ¿Cuál es el año en el que se celebraron la mayor cantidad de carreras? tmb erroes corregidos por chat gpt

query = ("""
        SELECT strftime('%Y', date) AS year, COUNT(*) AS total_races
FROM races
GROUP BY year
ORDER BY total_races DESC
LIMIT 1;
 """)

df_año_races = pd.read_sql_query(query, engine1)


In [67]:
df_año_races

,year,total_races
0,2024,24


In [ ]:
#Ejercicio 6: Pilotos con más de 50 Carreras
#¿Cuáles son los pilotos (por su driverId) que han participado en más de 50 carreras? mismo que los anteriores

query = ("""
         SELECT driverId, COUNT(*) AS total_carreras
FROM results
GROUP BY driverId
HAVING COUNT(*) > 50
ORDER BY total_carreras DESC;
 """)

driver_reces = pd.read_sql_query(query, engine1)

In [71]:
driver_reces

,driverId,total_carreras
0,4,404
1,1,356
2,8,352
3,22,326
4,18,309
...,...,...
154,403,52
155,456,51
156,305,51
157,262,51


si bien tengo una idea , todavia no logro darme cuenta cuando se pone la pregunta mas compleja, explicar de nuevo los conceptos basicos  y practicar 
mas ejercicos basicos , para empezar a darme cuenta de a poco .